# 10 Unmatched flags (T29)
Flagged runs of Model C on the test period that touch no labeled hour. Part A samples 30 of them and draws each with its surroundings (unmatched_flags_review.pdf) with a sheet for hand judgement. Part B tags all unmatched runs: exactly zero, site-wide, near a labeled event. Run after notebook 08.

In [ ]:
import sys
from config import *
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
M="C"; N_TOP=15; N_RAND=15; NEAR=24
f=pd.read_csv(OUT_DIR/"flags_ext.csv.gz",parse_dates=["timestamp"])
f=f[f.split=="test"].sort_values(["building_id","timestamp"]).reset_index(drop=True)
pr=pd.read_csv(OUT_DIR/"predictions.csv.gz",parse_dates=["timestamp"])[["building_id","timestamp","meter_reading","pred_C","pred_last_hour"]]
info=pd.read_csv(OUT_DIR/"labeled_buildings_info.csv").set_index("building_id")
f["t"]=(f.timestamp-pd.Timestamp("2016-01-01"))//pd.Timedelta(hours=1)
fl=f[f["flag_"+M]].copy()
gap=fl.groupby("building_id")["t"].diff(); fl["rid"]=(gap.isna()|(gap>1)).cumsum()
runs=fl.groupby("rid").agg(building_id=("building_id","first"),start_t=("t","min"),end_t=("t","max"),hours=("t","size"),hit=("label","max"),peak_score=("score_"+M,"max")).reset_index()
# nearest labeled hour distance
lab=f[f.label==1].groupby("building_id")["t"].apply(lambda s:s.to_numpy())
def near(b,s,e):
    a=lab.get(b)
    if a is None or len(a)==0: return np.nan
    i=np.searchsorted(a,s); d=[]
    if i<len(a): d.append(max(0,a[i]-e))
    if i>0: d.append(max(0,s-a[i-1]))
    return min(d)
runs["hours_to_nearest_label"]=[near(b,s,e) for b,s,e in zip(runs.building_id,runs.start_t,runs.end_t)]
runs["hit"]=runs["hit"].astype(bool)
un=runs[~runs.hit].copy()
un["near_label"]=un.hours_to_nearest_label<=NEAR
tot=len(runs);nun=len(un)
print("flagged runs",tot,"| touching a label",int(runs.hit.sum()),"| unmatched",nun,"| unmatched within 24h of a label",int(un.near_label.sum()),"| unmatched with a single flagged hour",int((un.hours==1).sum()))
print("unmatched run hours: median",un.hours.median(),"max",un.hours.max())
s_top=un.sort_values("peak_score",ascending=False).head(N_TOP)
rest=un.drop(s_top.index)
s_rand=rest.sample(min(N_RAND,len(rest)),random_state=0)
s_top=s_top.assign(group="top-ranked unmatched");s_rand=s_rand.assign(group="random unmatched")
S=pd.concat([s_top,s_rand]).reset_index(drop=True);S.insert(0,"review_id",range(1,len(S)+1))
S["start_time"]=pd.Timestamp("2016-01-01")+pd.to_timedelta(S.start_t,unit="h");S["end_time"]=pd.Timestamp("2016-01-01")+pd.to_timedelta(S.end_t,unit="h")
S["primary_use"]=S.building_id.map(info.primary_use);S["site_id"]=S.building_id.map(info.site_id)
pi=pr.set_index(["building_id","timestamp"])
with PdfPages(OUT_DIR/"unmatched_flags_review.pdf") as pdf:
    for k in range(0,len(S),4):
        fig,axs=plt.subplots(4,1,figsize=(11,13))
        for ax,(_,r) in zip(axs,S.iloc[k:k+4].iterrows()):
            lo=r.start_time-pd.Timedelta(days=3);hi=r.end_time+pd.Timedelta(days=3)
            w=pr[(pr.building_id==r.building_id)&(pr.timestamp>=lo)&(pr.timestamp<=hi)]
            ax.plot(w.timestamp,w.meter_reading,color="black",lw=1.2,label="actual")
            ax.plot(w.timestamp,w.pred_C,color="tab:green",lw=0.9,label="Model C expected")
            ax.axvspan(r.start_time,r.end_time+pd.Timedelta(hours=1),color="tab:red",alpha=0.25,label="flagged")
            lw=f[(f.building_id==r.building_id)&(f.label==1)&(f.timestamp>=lo)&(f.timestamp<=hi)]
            if len(lw): ax.scatter(lw.timestamp,np.full(len(lw),w.meter_reading.max()),marker="|",color="tab:blue",s=60,label="labeled unusual")
            ax.set_title(f"#{int(r.review_id)} ({r.group}) building {int(r.building_id)}, {r.primary_use}, site {int(r.site_id)}; {r.start_time:%d %b %H:%M}, {int(r.hours)} h flagged; score {r.peak_score:.1f}; {'within 24 h of a label' if r.near_label else 'no label within 24 h'}",fontsize=9)
            ax.legend(fontsize=7,loc="upper right");ax.set_ylabel("kWh")
        plt.tight_layout();pdf.savefig(fig);plt.close(fig)
out=S[["review_id","group","building_id","primary_use","site_id","start_time","end_time","hours","peak_score","near_label","hours_to_nearest_label"]].copy()
out["judgement (real unusual / plausible / false alarm / cannot tell)"]="";out["note"]=""
out.to_csv(OUT_DIR/"unmatched_flags_review_sheet.csv",index=False)
runs.assign(unmatched=~runs.hit).to_csv(OUT_DIR/"flagged_runs_model_C_test.csv",index=False)
print(out.head(8).to_string(index=False))


In [ ]:
# Part B: tag every flagged run: site-wide, exactly zero, near a labeled event
info=pd.read_csv(OUT_DIR/"labeled_buildings_info.csv").set_index("building_id")
f=pd.read_csv(OUT_DIR/"flags_ext.csv.gz",parse_dates=["timestamp"]);f=f[f.split=="test"].copy()
pr=pd.read_csv(OUT_DIR/"predictions.csv.gz",parse_dates=["timestamp"])[["building_id","timestamp","meter_reading","pred_C"]]
f=f.merge(pr,on=["building_id","timestamp"],how="left");f["t"]=(f.timestamp-pd.Timestamp("2016-01-01"))//pd.Timedelta(hours=1)
f["site"]=f.building_id.map(info.site_id)
g=f[f.valid].groupby(["site","timestamp"]).agg(nb=("building_id","nunique"),nfl=("flag_C","sum"));g["share"]=g.nfl/g.nb
sitewide_hours=set(g[(g.nb>=3)&(g.share>=.3)].index)   # site-hours where at least 3 buildings and 30% of them are flagged
runs=pd.read_csv(OUT_DIR/"flagged_runs_model_C_test.csv")
runs["site"]=runs.building_id.map(info.site_id);runs["start"]=pd.Timestamp("2016-01-01")+pd.to_timedelta(runs.start_t,unit="h")
runs["sitewide"]=[any((r.site,r.start+pd.Timedelta(hours=h)) in sitewide_hours for h in range(int(r.hours))) for r in runs.itertuples()]
fl=f[f.flag_C].set_index(["building_id","t"])
def cat(r):
    rows=fl.loc[[(r.building_id,t) for t in range(int(r.start_t),int(r.end_t)+1) if (r.building_id,t) in fl.index]]
    return pd.Series(dict(all_zero=bool((rows.meter_reading==0).all()),share_below_expected=round(float((rows.meter_reading<rows.pred_C).mean()),2)))
runs=pd.concat([runs.drop(columns=[c for c in ["all_zero","share_below_expected"] if c in runs]),runs.apply(cat,axis=1)],axis=1)
u=runs[~runs.hit.astype(bool)].copy()
u["why"]=np.where(u.all_zero,"reading is exactly zero",np.where(u.sitewide,"site-wide event",np.where(u.hours_to_nearest_label<=24,"within 24 h of a labeled event","other")))
print(u.why.value_counts().to_string())
runs.to_csv(OUT_DIR/"flagged_runs_model_C_test.csv",index=False);u.to_csv(OUT_DIR/"unmatched_runs_tagged.csv",index=False)
